## Step 1: Install Necessary Libraries

# Continued Pretraining vs Instruction Fine-Tuning

In this notebook, we are performing **continued pretraining / non-instruction fine-tuning** on raw pharma PDF text.

The model is given raw domain text such as:

> Metformin is one of the most widely prescribed oral antihyperglycemic agents...

The model then learns to **predict the next token** from this raw text.

This means the model learns:

- Pharma language
- Drug names
- Medical terminology
- Scientific writing style
- Domain-specific sentence patterns

However, the model is **not explicitly taught**:

- How to answer a user's question
- How to follow instructions
- How to respond in Q&A format
- How to behave like a domain-specific chatbot

---

## Step 2: Import Libraries

In [2]:
import os
import re
import gc
import math
import json
import random
import unicodedata
import fitz  # PyMuPDF
import torch
import inspect
from dataclasses import dataclass, asdict
from typing import List, Dict, Any
from datasets import Dataset, DatasetDict, load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, DataCollatorForLanguageModeling,
    Trainer, TrainingArguments, set_seed,
)
from peft import (
    LoraConfig, TaskType, get_peft_model,
    prepare_model_for_kbit_training, PeftModel,
)
from trl import DPOTrainer, DPOConfig
from google.colab import userdata
from huggingface_hub import HfApi
import warnings
warnings.filterwarnings("ignore")

## Step 3: Global configuration

In [ ]:
@dataclass
class NonInstructionConfig:
    # Path of the pharma PDF file that will be used as the raw domain corpus.
    pdf_path: str = "/data/Metformin-Lipid-Therapy-Knowledge.pdf"

    # Base causal language model that we will fine-tune on pharma-domain text.
    model_name: str = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

    # Directory where training checkpoints will be saved during fine-tuning.
    output_dir: str = "/output/pharma_tinyllama_lora_output"

    # Directory where the final trained LoRA adapter will be saved.
    adapter_dir: str = "/output/pharma_tinyllama_lora_adapter"

    # Directory where the final merged (adapter + base) model will be saved.
    merged_model_dir = "/output/pharma_tinyllama_merged_model"

    # Directory where cleaned and processed training data will be saved.
    processed_data_dir: str = "/output/pharma_processed_data"

    # Minimum paragraph length required to keep a paragraph for training.
    min_chars_per_paragraph: int = 80

    # Number of tokens in each training block for causal language modeling.
    block_size: int = 512

    # Percentage of data used for validation instead of training.
    test_size: float = 0.15

    # Random seed used to make splitting and training more reproducible.
    seed: int = 42

    # LoRA rank; controls the size and capacity of the trainable adapter.
    lora_r: int = 16

    # LoRA scaling factor; controls the strength of the LoRA update.
    lora_alpha: int = 32

    # Dropout applied inside LoRA layers to reduce overfitting.
    lora_dropout: float = 0.05

    # Number of times the model will see the complete training dataset.
    num_train_epochs: float = 3.0

    # Number of training samples processed per GPU/device at one time.
    per_device_train_batch_size: int = 1

    # Number of validation samples processed per GPU/device at one time.
    per_device_eval_batch_size: int = 1

    # Number of small batches accumulated before one optimizer update.
    gradient_accumulation_steps: int = 8

    # Step size used by the optimizer to update trainable LoRA weights.
    learning_rate: float = 2e-4

    # Fraction of early training steps used to gradually increase learning rate.
    warmup_ratio: float = 0.03

    # Regularization value used to prevent weights from becoming too large.
    weight_decay: float = 0.01

    # Number of training steps after which logs will be printed.
    logging_steps=1
    logging_first_step=True

    # Number of training steps after which validation will be performed.
    eval_steps: int = 10

    # Number of training steps after which a checkpoint will be saved.
    save_steps: int = 25

    # Maximum number of checkpoints to keep; older checkpoints will be deleted.
    save_total_limit: int = 2

    # Maximum number of training steps; -1 means train using num_train_epochs.
    max_steps: int = -1

non_instruction_config = NonInstructionConfig()

In [ ]:
@dataclass
class InstructionConfig:
    # Path to the instruction dataset in JSONL format.
    # Each line should look like:
    # {"instruction": "...", "input": "", "output": "..."}
    instruction_data_path: str = "/data/pharma_instruction_dataset.jsonl"

    # Base causal language model that we will instruction fine-tune.
    model_name: str = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

    # Directory where training checkpoints will be saved during fine-tuning.
    output_dir: str = "/output/pharma_tinyllama_instruction_lora_output"

    # Directory where the final trained LoRA adapter will be saved.
    adapter_dir: str = "/output/pharma_tinyllama_instruction_lora_adapter"

    # Directory where the final merged (adapter + base) model will be saved.
    merged_model_dir: str = "/output/pharma_tinyllama_instruction_merged_model"

    # Maximum sequence length used during tokenization.
    max_length: int = 512

    # Percentage of data used for validation instead of training.
    test_size: float = 0.15

    # Random seed used to make splitting and training more reproducible.
    seed: int = 42

    # LoRA rank; controls the size and capacity of the trainable adapter.
    lora_r: int = 16

    # LoRA scaling factor; controls the strength of the LoRA update.
    lora_alpha: int = 32

    # Dropout applied inside LoRA layers to reduce overfitting.
    lora_dropout: float = 0.05

    # Number of times the model will see the complete training dataset.
    num_train_epochs: float = 5.0

    # Number of training samples processed per GPU/device at one time.
    per_device_train_batch_size: int = 1

    # Number of validation samples processed per GPU/device at one time.
    per_device_eval_batch_size: int = 1

    # Number of small batches accumulated before one optimizer update.
    gradient_accumulation_steps: int = 8

    # Step size used by the optimizer to update trainable LoRA weights.
    learning_rate: float = 1e-4

    # Number of warmup steps used to gradually increase the learning rate.
    warmup_steps: int = 5

    # Regularization value used to prevent weights from becoming too large.
    weight_decay: float = 0.01

    # Number of training steps after which logs will be printed.
    logging_steps: int = 1
    logging_first_step: bool = True

    # Number of training steps after which validation will be performed.
    eval_steps: int = 1

    # Number of training steps after which a checkpoint will be saved.
    save_steps: int = 10

    # Maximum number of checkpoints to keep; older checkpoints will be deleted.
    save_total_limit: int = 2

    # Maximum number of training steps; -1 means train using num_train_epochs.
    max_steps: int = -1

instruction_config = InstructionConfig()
instruction_config


InstructionConfig(instruction_data_path='/content/pharma_instruction_dataset.jsonl', model_name='TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T', output_dir='/content/pharma_tinyllama_instruction_lora_output', adapter_dir='/content/pharma_tinyllama_instruction_lora_adapter', merged_model_dir='/content/pharma_tinyllama_instruction_merged_model', max_length=512, test_size=0.15, seed=42, lora_r=16, lora_alpha=32, lora_dropout=0.05, num_train_epochs=5.0, per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=8, learning_rate=0.0001, warmup_steps=5, weight_decay=0.01, logging_steps=1, logging_first_step=True, eval_steps=1, save_steps=10, save_total_limit=2, max_steps=-1)

In [ ]:
@dataclass
class PreferenceConfig:
    # Path to the preference dataset in JSONL format.
    # Each line should look like:
    # {"prompt": "...", "chosen": "...", "rejected": "..."}
    preference_data_path: str = "/data/pharma_preference_dataset.jsonl"

    # Base (or instruction-tuned) causal language model to preference-tune.
    # Point this at a local instruction-tuned model directory if you have one.
    model_name: str = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

    # Directory where training checkpoints will be saved during DPO training.
    output_dir: str = "/output/pharma_tinyllama_preference_dpo_output"

    # Directory where the final trained DPO LoRA adapter will be saved.
    adapter_dir: str = "/output/pharma_tinyllama_preference_dpo_lora_adapter"

    # Directory where the final merged (adapter + base) model will be saved.
    merged_model_dir: str = "/output/pharma_tinyllama_preference_merged_model"

    # Percentage of data used for validation instead of training.
    test_size: float = 0.15

    # Random seed used to make splitting and training more reproducible.
    seed: int = 42

    # LoRA rank; controls the size and capacity of the trainable adapter.
    lora_r: int = 16

    # LoRA scaling factor; controls the strength of the LoRA update.
    lora_alpha: int = 32

    # Dropout applied inside LoRA layers to reduce overfitting.
    lora_dropout: float = 0.05

    # Number of times the model will see the complete training dataset.
    num_train_epochs: float = 3.0

    # Number of training samples processed per GPU/device at one time.
    per_device_train_batch_size: int = 1

    # Number of validation samples processed per GPU/device at one time.
    per_device_eval_batch_size: int = 1

    # Number of small batches accumulated before one optimizer update.
    # NOTE: with small preference datasets, keep this <= your number of
    # training examples, otherwise the Trainer will silently need multiple
    # passes over the data to complete a single optimizer step, inflating
    # the effective number of epochs actually run.
    gradient_accumulation_steps: int = 4

    # Step size used by the optimizer to update trainable LoRA weights.
    learning_rate: float = 5e-5

    # Number of warmup steps used to gradually increase the learning rate.
    warmup_steps: int = 2

    # Regularization value used to prevent weights from becoming too large.
    weight_decay: float = 0.01

    # Number of training steps after which logs will be printed.
    logging_steps: int = 1
    logging_first_step: bool = True

    # Number of training steps after which validation will be performed.
    eval_steps: int = 1

    # Number of training steps after which a checkpoint will be saved.
    save_steps: int = 10

    # Maximum number of checkpoints to keep; older checkpoints will be deleted.
    save_total_limit: int = 2

    # Maximum number of training steps; -1 means train using num_train_epochs.
    max_steps: int = -1

    # DPO-specific hyperparameter: controls how strongly the model is
    # pushed toward chosen answers over rejected answers (vs. the
    # reference model). Common values: 0.1 (default-ish), up to ~0.5.
    beta: float = 0.1

    # Maximum total sequence length (prompt + response) for DPO.
    max_length: int = 512

    # Maximum prompt length (truncated from the left if longer).
    max_prompt_length: int = 256

preference_config = PreferenceConfig()
preference_config

PreferenceConfig(preference_data_path='/content/pharma_preference_dataset.jsonl', model_name='TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T', output_dir='/content/pharma_tinyllama_preference_dpo_output', adapter_dir='/content/pharma_tinyllama_preference_dpo_lora_adapter', merged_model_dir='/content/pharma_tinyllama_preference_merged_model', test_size=0.15, seed=42, lora_r=16, lora_alpha=32, lora_dropout=0.05, num_train_epochs=3.0, per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=4, learning_rate=5e-05, warmup_steps=2, weight_decay=0.01, logging_steps=1, logging_first_step=True, eval_steps=1, save_steps=10, save_total_limit=2, max_steps=-1, beta=0.1, max_length=512, max_prompt_length=256)

In [6]:
HF_USERNAME = "saadtariq"

BASE_MODEL_NAME = non_instruction_config.model_name

# Stage 1: Non-instruction LoRA adapter
HF_REPO_NON_INSTRUCTION_ADAPTER = f"{HF_USERNAME}/pharma-tinyllama-non-instruction-lora-adapter"

# Stage 1 merged model
HF_REPO_NON_INSTRUCTION_MERGED = f"{HF_USERNAME}/pharma-tinyllama-non-instruction-merged"

# Stage 2: Instruction LoRA adapter
HF_REPO_INSTRUCTION_ADAPTER = f"{HF_USERNAME}/pharma-tinyllama-instruction-lora-adapter"

# Stage 2 merged model
HF_REPO_INSTRUCTION_MERGED = f"{HF_USERNAME}/pharma-tinyllama-instruction-merged"

# Stage 3: DPO preference LoRA adapter
HF_REPO_DPO_ADAPTER = f"{HF_USERNAME}/pharma-tinyllama-dpo-lora-adapter"

# Stage 3 final merged model
HF_REPO_DPO_MERGED = f"{HF_USERNAME}/pharma-tinyllama-dpo-merged"

print(HF_REPO_NON_INSTRUCTION_ADAPTER)
print(HF_REPO_INSTRUCTION_ADAPTER)
print(HF_REPO_DPO_ADAPTER)

saadtariq/pharma-tinyllama-non-instruction-lora-adapter
saadtariq/pharma-tinyllama-instruction-lora-adapter
saadtariq/pharma-tinyllama-dpo-lora-adapter


# Stage 1: Non-instruction Causal LLM Fine-tuning or Domain-Adaptive Continued Pretraining

## Pipeline

```text
Pharma PDF
   ↓
PDF text extraction
   ↓
Text cleaning and normalization
   ↓
Data creation
   ↓
Hugging Face Dataset Conversion
   ↓
Tokenization
   ↓
LoRA/QLoRA fine-tuning
   ↓
Validation loss
   ↓
Adapter saving and reloading
   ↓
Text continuation inference
```

In [7]:
non_instruction_config

NonInstructionConfig(pdf_path='/content/Metformin-Lipid-Therapy-Knowledge.pdf', model_name='TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T', output_dir='/content/pharma_tinyllama_lora_output', adapter_dir='/content/pharma_tinyllama_lora_adapter', processed_data_dir='/content/pharma_processed_data', min_chars_per_paragraph=80, block_size=512, test_size=0.15, seed=42, lora_r=16, lora_alpha=32, lora_dropout=0.05, num_train_epochs=3.0, per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=8, learning_rate=0.0002, warmup_ratio=0.03, weight_decay=0.01, eval_steps=10, save_steps=25, save_total_limit=2, max_steps=-1)

In [8]:
print(json.dumps(asdict(non_instruction_config), indent=2))

{
  "pdf_path": "/content/Metformin-Lipid-Therapy-Knowledge.pdf",
  "model_name": "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
  "output_dir": "/content/pharma_tinyllama_lora_output",
  "adapter_dir": "/content/pharma_tinyllama_lora_adapter",
  "processed_data_dir": "/content/pharma_processed_data",
  "min_chars_per_paragraph": 80,
  "block_size": 512,
  "test_size": 0.15,
  "seed": 42,
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "num_train_epochs": 3.0,
  "per_device_train_batch_size": 1,
  "per_device_eval_batch_size": 1,
  "gradient_accumulation_steps": 8,
  "learning_rate": 0.0002,
  "warmup_ratio": 0.03,
  "weight_decay": 0.01,
  "eval_steps": 10,
  "save_steps": 25,
  "save_total_limit": 2,
  "max_steps": -1
}


In [9]:
non_instruction_config.output_dir

'/content/pharma_tinyllama_lora_output'

In [10]:
non_instruction_config.processed_data_dir

'/content/pharma_processed_data'

In [11]:
os.makedirs(non_instruction_config.output_dir, exist_ok=True)
os.makedirs(non_instruction_config.adapter_dir, exist_ok=True)
os.makedirs(non_instruction_config.processed_data_dir, exist_ok=True)
os.makedirs(non_instruction_config.merged_model_dir, exist_ok=True)

## Step 4: Optional Colab upload helper

In [12]:
if not os.path.exists(non_instruction_config.pdf_path):
    print(f"PDF not found at: {non_instruction_config.pdf_path}")
else:
    print(f"PDF found: {non_instruction_config.pdf_path}")

PDF found: /content/Metformin-Lipid-Therapy-Knowledge.pdf


## Step 5 - Stage 1.1: Extract text from PDF

In [13]:
def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:
    # Extract page-level text from a PDF.
    pages = []
    with fitz.open(pdf_path) as doc:
        for page_index, page in enumerate(doc, start=1):
            text = page.get_text("text")
            text = text.strip() if text else ""
            if text:
                pages.append({
                    "page": page_index,
                    "text": text,
                    "char_count": len(text),
                })
    return pages

In [14]:
non_instruction_config.pdf_path

'/content/Metformin-Lipid-Therapy-Knowledge.pdf'

In [15]:
pdf_pages = extract_pdf_pages(non_instruction_config.pdf_path)

In [16]:
print(f"Total pages with extracted text: {len(pdf_pages)}")
print("Page-level character counts:")
for item in pdf_pages:
    print(f"Page {item['page']}: {item['char_count']} characters")

Total pages with extracted text: 6
Page-level character counts:
Page 1: 2244 characters
Page 2: 2889 characters
Page 3: 2636 characters
Page 4: 2416 characters
Page 5: 2613 characters
Page 6: 2761 characters


In [17]:
print(pdf_pages[0]["text"])

Metformin is one of the most widely prescribed oral antihyperglycemic agents.​
 Its primary mechanism of action involves the activation of AMP-activated protein kinase 
(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation 
while inhibiting hepatic gluconeogenesis.​
 Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes 
and display anti-inflammatory properties.​
 Recent studies also suggest potential anticancer effects through inhibition of the mTOR 
signaling pathway and suppression of tumor angiogenesis. 
 
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in 
significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to 
monotherapy.​
 Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal 
wall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA reductase, 
suppressing endogenous cho

| Cleaning Step                          | Code / Logic                             | What It Does                                                                  | Example Before                                                  | Example After                                                  | Why It Matters for Fine-Tuning                                            |
| -------------------------------------- | ---------------------------------------- | ----------------------------------------------------------------------------- | --------------------------------------------------------------- | -------------------------------------------------------------- | ------------------------------------------------------------------------- |
| Unicode normalization                  | `unicodedata.normalize("NFKC", text)`    | Converts unusual Unicode characters into standard readable characters.        | `ＡＭＰＫ`, `ﬁ`                                                     | `AMPK`, `fi`                                                   | Prevents tokenizer confusion caused by hidden or non-standard characters. |
| Remove zero-width characters           | `text.replace("\u200b", "")`             | Removes invisible zero-width spaces from PDF text.                            | `Metformin​ activates AMPK`                                     | `Metformin activates AMPK`                                     | Invisible characters can create bad tokens and noisy training data.       |
| Remove BOM / hidden marker             | `text.replace("\ufeff", "")`             | Removes hidden Byte Order Mark characters sometimes found in extracted text.  | `﻿Metformin is used...`                                         | `Metformin is used...`                                         | Keeps the training text clean and consistent.                             |
| Fix hyphenated line breaks             | `re.sub(r"(\w)-\n(\w)", r"\1\2", text)`  | Joins words that were broken across PDF lines.                                | `gluconeogene-\nsis`                                            | `gluconeogenesis`                                              | Prevents the model from learning broken medical terms.                    |
| Normalize spaces and tabs              | `re.sub(r"[ \t]+", " ", text)`           | Converts multiple spaces or tabs into one space.                              | `Metformin     activates    AMPK`                               | `Metformin activates AMPK`                                     | Makes text consistent and easier for tokenizer/model to learn.            |
| Normalize blank lines                  | `re.sub(r"\n{3,}", "\n\n", text)`        | Converts too many blank lines into a proper paragraph gap.                    | `Para 1\n\n\n\nPara 2`                                          | `Para 1\n\nPara 2`                                             | Preserves paragraph structure without unnecessary whitespace noise.       |
| Remove standalone page numbers         | `re.sub(r"(?m)^\s*\d+\s*$", "", text)`   | Removes lines that contain only page numbers.                                 | `1` or `23`                                                     | Removed                                                        | Prevents the model from learning irrelevant PDF page numbers.             |
| Split into paragraphs                  | `re.split(r"\n\s*\n", text)`             | Splits text wherever there is a blank line.                                   | `Para 1\n\nPara 2`                                              | `["Para 1", "Para 2"]`                                         | Helps preserve meaningful document structure.                             |
| Remove line wrapping inside paragraphs | `re.sub(r"\n+", " ", paragraph)`         | Converts broken lines inside the same paragraph into a single paragraph line. | `Metformin is widely prescribed\noral antihyperglycemic agent.` | `Metformin is widely prescribed oral antihyperglycemic agent.` | Prevents the model from learning artificial PDF line breaks.              |
| Normalize paragraph spacing            | `re.sub(r"\s+", " ", paragraph).strip()` | Removes extra spaces inside each paragraph and trims start/end spaces.        | `  Metformin   activates   AMPK.  `                             | `Metformin activates AMPK.`                                    | Produces clean, readable training examples.                               |
| Remove empty paragraphs                | `if paragraph:`                          | Keeps only non-empty cleaned paragraphs.                                      | `""`                                                            | Removed                                                        | Avoids useless blank samples in the dataset.                              |
| Rebuild cleaned text                   | `"\n\n".join(cleaned_paragraphs)`        | Joins cleaned paragraphs with two newlines.                                   | List of cleaned paragraphs                                      | Clean paragraph-level text                                     | Creates a clean corpus suitable for causal LM training.                   |
| Track cleaned page length              | `char_count: len(cleaned_text)`          | Stores number of characters after cleaning.                                   | Raw page length unknown                                         | `char_count = 1450`                                            | Helps debug whether a page has too little or too much extracted content.  |
| Preview cleaned output                 | `cleaned_pages[0]["text"][:1500]`        | Prints first 1500 characters of cleaned page 1.                               | Full cleaned page                                               | Preview text                                                   | Helps manually verify that cleaning worked correctly.                     |


## Step 6 - Stage 1.2: Text cleaning utilities

In [18]:
def clean_pdf_text(text: str) -> str:
    # Standardize Unicode text so visually similar characters are treated consistently.
    # Example: "ＡＭＰＫ" becomes "AMPK" and "ﬁ" becomes "fi".
    text = unicodedata.normalize("NFKC", text)

    # Remove invisible characters that may appear during PDF text extraction.
    text = text.replace("\u200b", "").replace("\ufeff", "")

    # Join words broken by line hyphenation, e.g., "gluconeogene-\nsis" -> "gluconeogenesis".
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Replace multiple spaces/tabs with a single space.
    text = re.sub(r"[ \t]+", " ", text)

    # Convert three or more newlines into a standard paragraph break.
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove lines that contain only page numbers.
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    # Split text into paragraphs, clean each paragraph, and remove empty ones.
    paragraphs = []
    for paragraph in re.split(r"\n\s*\n", text):
        paragraph = re.sub(r"\n+", " ", paragraph)
        paragraph = re.sub(r"\s+", " ", paragraph).strip()

        if paragraph:
            paragraphs.append(paragraph)

    # Join cleaned paragraphs with one blank line between them.
    return "\n\n".join(paragraphs)

In [19]:
cleaned_pages = []

for page in pdf_pages:
    cleaned_text = clean_pdf_text(page["text"])
    cleaned_pages.append({
        "page": page["page"],
        "text": cleaned_text,
        "char_count": len(cleaned_text),
    })

print("Total cleaned pages:", len(cleaned_pages))

Total cleaned pages: 6


In [20]:
print(cleaned_pages[0]["text"])

Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties. Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis.

Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to monotherapy. Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal wall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA reductase, suppressing endogenous cholesterol synthesis

## Step 7 - Stage 1.3: Split cleaned pages into paragraphs

In [21]:
def split_into_paragraph_records(cleaned_pages, min_chars=80):
    paragraph_records = []

    for page in cleaned_pages:
        # Split page text into paragraphs using blank lines.
        paragraphs = page["text"].split("\n\n")

        for paragraph_index, paragraph in enumerate(paragraphs, start=1):
            # Remove extra spaces from the beginning and end.
            paragraph = paragraph.strip()

            # Skip very short paragraphs because they are usually headings, page numbers, or noise.
            if len(paragraph) < min_chars:
                continue

            # Store each useful paragraph with basic metadata.
            paragraph_records.append({
                "text": paragraph,
                "source_page": page["page"],
                "paragraph_id": paragraph_index,
                "char_count": len(paragraph),
            })

    return paragraph_records

In [22]:
paragraph_records = split_into_paragraph_records(cleaned_pages)

In [23]:
print("Total paragraph records:", len(paragraph_records))

Total paragraph records: 9


In [24]:
for record in paragraph_records[:3]:
    print("=" * 80)
    print(f"Page: {record['source_page']} | Paragraph: {record['paragraph_id']} | Characters: {record['char_count']}")
    print(record["text"])

Page: 1 | Paragraph: 1 | Characters: 575
Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties. Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis.
Page: 1 | Paragraph: 2 | Characters: 598
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to monotherapy. Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal wall, reducing cholesterol absorption, while Atorvastatin

## Stage 8 - Step 1.4: Save extracted and cleaned corpus for auditability

In [25]:
raw_pages_path = os.path.join(
    non_instruction_config.processed_data_dir,
    "pdf_pages_raw.jsonl"
)

paragraphs_path = os.path.join(
    non_instruction_config.processed_data_dir,
    "pharma_paragraph_process.jsonl"
)

In [26]:
with open(raw_pages_path, "w", encoding="utf-8") as f:
    for item in pdf_pages:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open(paragraphs_path, "w", encoding="utf-8") as f:
    for item in paragraph_records:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved raw pages to: {raw_pages_path}")
print(f"Saved cleaned paragraph corpus to: {paragraphs_path}")

Saved raw pages to: /content/pharma_processed_data/pdf_pages_raw.jsonl
Saved cleaned paragraph corpus to: /content/pharma_processed_data/pharma_paragraph_process.jsonl


## Step 9 - Stage 1.5: Create Hugging Face Dataset

In [27]:
if len(paragraph_records) < 2:
    raise ValueError(
        "The extracted corpus is too small. Please provide a larger pharma PDF or lower min_chars_per_paragraph."
    )
text_dataset = Dataset.from_list(paragraph_records)

In [28]:
print(text_dataset)

Dataset({
    features: ['text', 'source_page', 'paragraph_id', 'char_count'],
    num_rows: 9
})


In [29]:
print(text_dataset[0])

{'text': 'Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties. Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis.', 'source_page': 1, 'paragraph_id': 1, 'char_count': 575}


## Step 10 - Stage 1.6: Train/Eval Split

In [30]:
split_dataset = text_dataset.train_test_split(
    test_size=non_instruction_config.test_size,
    seed=non_instruction_config.seed
)

dataset = DatasetDict({
    "train": split_dataset["train"],
    "validation": split_dataset["test"],
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'source_page', 'paragraph_id', 'char_count'],
        num_rows: 7
    })
    validation: Dataset({
        features: ['text', 'source_page', 'paragraph_id', 'char_count'],
        num_rows: 2
    })
})


## Step 11 - Stage 1.7: Load Tokenizer

In [31]:
tokenizer = AutoTokenizer.from_pretrained(
    non_instruction_config.model_name, use_fast=True
)

# Some Llama-style models do not define a pad token.
# For causal LM fine-tuning, using EOS as PAD is a common practical choice.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [32]:
tokenizer.eos_token

'</s>'

In [33]:
print(f"Tokenizer loaded: {non_instruction_config.model_name}")
print(f"Vocab size: {len(tokenizer)}")
print(f"Pad token: {tokenizer.pad_token} | Pad token id: {tokenizer.pad_token_id}")
print(f"EOS token: {tokenizer.eos_token} | EOS token id: {tokenizer.eos_token_id}")

Tokenizer loaded: TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T
Vocab size: 32000
Pad token: </s> | Pad token id: 2
EOS token: </s> | EOS token id: 2


## Step 12 - Stage 1.8: Tokenization and Text Packing

In [34]:
def tokenize_function(examples):
    # Tokenize text without padding. Padding is handled dynamically by the collator.
    return tokenizer(examples["text"])

In [35]:
tokenized_datasets = dataset.map(
    tokenize_function,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing text corpus",
)

Tokenizing text corpus:   0%|          | 0/7 [00:00<?, ? examples/s]

Tokenizing text corpus:   0%|          | 0/2 [00:00<?, ? examples/s]

| Parameter                                       | Meaning                                                                                                                               |
| ----------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------- |
| `tokenize_function`                             | This function converts each text example into token IDs.                                                                              |
| `batched=True`                                  | The function processes multiple rows at once instead of one row at a time. This makes tokenization faster.                            |
| `remove_columns=datasets["train"].column_names` | After tokenization, the original dataset columns are removed. Only tokenized columns such as `input_ids` and `attention_mask` remain. |
| `desc="Tokenizing text corpus"`                 | This message is shown in the progress bar so we can understand that tokenization is currently running.                                |


In [36]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 7
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 2
    })
})

In [38]:
# tokenized_datasets['train']['input_ids'][0]

In [39]:
def create_training_blocks(tokenized_examples):
    # Join all token IDs from multiple examples into one long list.
    all_input_ids = []
    all_attention_masks = []

    for input_ids in tokenized_examples["input_ids"]:
        all_input_ids.extend(input_ids)

    for attention_mask in tokenized_examples["attention_mask"]:
        all_attention_masks.extend(attention_mask)

    # Calculate how many complete blocks we can create.
    total_tokens = len(all_input_ids)
    usable_tokens = (total_tokens // non_instruction_config.block_size) * non_instruction_config.block_size

    # If we do not have enough tokens to create even one block, return empty data.
    if usable_tokens == 0:
        return {
            "input_ids": [],
            "attention_mask": [],
            "labels": [],
        }

    # Keep only tokens that can fit into complete fixed-size blocks.
    all_input_ids = all_input_ids[:usable_tokens]
    all_attention_masks = all_attention_masks[:usable_tokens]

    # Split the long token list into fixed-size training blocks.
    input_id_blocks = []
    attention_mask_blocks = []

    for start_index in range(0, usable_tokens, non_instruction_config.block_size):
        end_index = start_index + non_instruction_config.block_size

        input_id_blocks.append(all_input_ids[start_index:end_index])
        attention_mask_blocks.append(all_attention_masks[start_index:end_index])

    # For causal language modeling, labels are the same as input IDs.
    # The model uses these labels to learn next-token prediction.
    labels = input_id_blocks.copy()

    return {
        "input_ids": input_id_blocks,
        "attention_mask": attention_mask_blocks,
        "labels": labels,
    }

In [40]:
final_dataset = tokenized_datasets.map(
    create_training_blocks,
    batched=True,
    desc=f"Creating fixed-size training blocks of {non_instruction_config.block_size} tokens",
)

Creating fixed-size training blocks of 512 tokens:   0%|          | 0/7 [00:00<?, ? examples/s]

Creating fixed-size training blocks of 512 tokens:   0%|          | 0/2 [00:00<?, ? examples/s]

In [41]:
sample = final_dataset["train"][0]

In [42]:
print("Keys:", sample.keys())
print("input_ids length:", len(sample["input_ids"]))
print("labels length:", len(sample["labels"]))
print("Decoded sample preview:\n")
print(tokenizer.decode(sample["input_ids"][:250]))

Keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
input_ids length: 512
labels length: 512
Decoded sample preview:

<s> Pharma Domain Training Data - Page 5 Page 5 - AI in Drug Discovery and Pharmaceutical R&D; Pharma-domain corpus extension for custom fine-tuning and RAG experimentation. Educational content only; not medical advice. Target identification Artificial intelligence is increasingly used in pharmaceutical research to analyze genomics, transcriptomics, proteomics, disease phenotypes, chemical libraries, and clinical datasets. In target identification, machine learning models can prioritize genes or proteins that may play causal roles in disease biology. These predictions are strengthened when integrated with experimental validation, pathway analysis, human genetics, and disease-relevant biomarkers. Molecular screening In early discovery, deep learning can support virtual screening by predicting protein-ligand binding affinity, molecular properties, toxicity signals,

## Step 13 - Stage 1.9: Load Model for QLoRA Training

In this step, we load the base model for fine-tuning.

If GPU is available, we load the model in **4-bit mode**.

This helps because:

- It uses less GPU memory
- It allows us to fine-tune larger models on limited hardware
- It is useful for Colab or small GPU environments
- It works well with LoRA/QLoRA fine-tuning

If GPU is not available, the model will load normally on CPU, but training will be much slower.

In [43]:
use_cuda = torch.cuda.is_available()
print("CUDA available:", use_cuda)
if use_cuda:
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [44]:
# Clear memory before loading the model.
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

In [45]:
if use_cuda:
    # Configure 4-bit quantization to reduce GPU memory usage.
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    # Load the base model in 4-bit mode on available GPU devices.
    base_model = AutoModelForCausalLM.from_pretrained(
        non_instruction_config.model_name,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True,
    )

    # Prepare the quantized model for stable LoRA/QLoRA training.
    base_model = prepare_model_for_kbit_training(base_model)

else:
    # Load the base model normally when GPU is not available.
    base_model = AutoModelForCausalLM.from_pretrained(
        non_instruction_config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

# Disable cache during training to reduce memory usage and avoid training warnings.
base_model.config.use_cache = False

print("Base model loaded successfully.")

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.40GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

Base model loaded successfully.


## Step 14 - Stage 1.10: Apply LoRA adapters

In [46]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=non_instruction_config.lora_r,
    lora_alpha=non_instruction_config.lora_alpha,
    lora_dropout=non_instruction_config.lora_dropout,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

In [47]:
model = get_peft_model(base_model, lora_config)

In [48]:
model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


## Step 15 - Stage 1.11: Data collator

### Why Do We Need `DataCollatorForLanguageModeling`?

After tokenization and text packing, our dataset contains token IDs in a training-ready structure.

However, the `Trainer` still needs a component that can take multiple examples from the dataset and convert them into a proper batch during training.

That component is called a **data collator**.

```python
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)
What Does the Data Collator Do?

The data collator prepares mini-batches for the model.

It handles things like:

Collecting multiple training examples together
Padding sequences if required
Converting examples into tensors
Preparing labels for language modeling
Example

Suppose our packed dataset has training examples like this:

Example 1 = 512 tokens
Example 2 = 512 tokens
Example 3 = 512 tokens

During training, the Trainer may take two examples at a time:

Batch = Example 1 + Example 2

The data collator converts them into tensors like:

input_ids shape      = [2, 512]
attention_mask shape = [2, 512]
labels shape         = [2, 512]

This is the format the model expects during training.

Why mlm=False?

mlm means Masked Language Modeling.

Masked Language Modeling is used for BERT-style models.

Example:

Metformin is used for [MASK].

The model predicts the masked word:

diabetes

But we are using TinyLlama, which is a causal language model.

Causal language models learn by predicting the next token from left to right.

Example:

Metformin → is
Metformin is → used
Metformin is used → for
Metformin is used for → diabetes

So we set:

mlm=False

This tells Hugging Face:

Do not use BERT-style masked language modeling. Use causal language modeling instead.

Why Is This Needed Even After Tokenization and Packing?

Tokenization converts text into token IDs.

Text packing groups token IDs into fixed-size blocks.

But the data collator prepares those blocks into actual training batches.

So the flow is:

Raw pharma text
   ↓
Tokenization
   ↓
Token IDs
   ↓
Text packing
   ↓
Fixed-size training blocks
   ↓
Data collator
   ↓
Mini-batches for Trainer
   ↓
Model training

In [49]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

## Step 16 - Stage 1.12: Training arguments

In [50]:
training_kwargs = dict(
    output_dir=non_instruction_config.output_dir,
    num_train_epochs=non_instruction_config.num_train_epochs,
    max_steps=non_instruction_config.max_steps,
    per_device_train_batch_size=non_instruction_config.per_device_train_batch_size,
    per_device_eval_batch_size=non_instruction_config.per_device_eval_batch_size,
    gradient_accumulation_steps=non_instruction_config.gradient_accumulation_steps,
    learning_rate=non_instruction_config.learning_rate,
    warmup_steps=5,
    weight_decay=non_instruction_config.weight_decay,

    # Log training loss at every step for small demo datasets.
    logging_steps=1,
    logging_first_step=True,

    eval_steps=non_instruction_config.eval_steps,
    save_steps=non_instruction_config.save_steps,
    save_total_limit=non_instruction_config.save_total_limit,
    fp16=use_cuda,
    bf16=False,
    report_to="none",
    remove_unused_columns=False,
)

In [51]:
training_args = TrainingArguments(**training_kwargs)

In [53]:
# print(training_args)

## Step 17: Stage 1.13: Build Trainer

In [54]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_dataset["train"],
    eval_dataset=final_dataset["validation"],
    data_collator=data_collator,
)
print("Trainer is ready.")

Trainer is ready.


## Step 18: Stage 1.14: Start Training

In [55]:
train_result = trainer.train()
print("Training completed.")

Step,Training Loss
1,2.148971
2,2.148971
3,2.123914


Training completed.


In [56]:
for log in trainer.state.log_history:
    print(log)

{'loss': 2.14897084236145, 'grad_norm': 0.6727555990219116, 'learning_rate': 0.0, 'epoch': 1.0, 'step': 1}
{'loss': 2.1489710807800293, 'grad_norm': 0.6843518614768982, 'learning_rate': 4e-05, 'epoch': 2.0, 'step': 2}
{'loss': 2.123913526535034, 'grad_norm': 0.6602457165718079, 'learning_rate': 8e-05, 'epoch': 3.0, 'step': 3}
{'train_runtime': 29.0272, 'train_samples_per_second': 0.62, 'train_steps_per_second': 0.103, 'total_flos': 57901993426944.0, 'train_loss': 2.1406184832255044, 'epoch': 3.0, 'step': 3}


## Step 19: Stage 1.15: Save Adapter and Tokenizer

In [57]:
trainer.model.save_pretrained(non_instruction_config.adapter_dir)
tokenizer.save_pretrained(non_instruction_config.adapter_dir)

print(f"LoRA adapter saved to: {non_instruction_config.adapter_dir}")
print("Saved files:")
print(os.listdir(non_instruction_config.adapter_dir))

LoRA adapter saved to: /content/pharma_tinyllama_lora_adapter
Saved files:
['adapter_config.json', 'adapter_model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'README.md']


## Step 20: Stage 1.16: Push Stage 1 non-instruction LoRA adapter to Hugging Face

In [58]:
HF_REPO_NON_INSTRUCTION_ADAPTER

'saadtariq/pharma-tinyllama-non-instruction-lora-adapter'

In [61]:
trainer.model.push_to_hub(
    HF_REPO_NON_INSTRUCTION_ADAPTER,
    private=True,
)

tokenizer.push_to_hub(
    HF_REPO_NON_INSTRUCTION_ADAPTER,
    private=True,
)

print("Stage 1 non-instruction LoRA adapter pushed to:")
print(HF_REPO_NON_INSTRUCTION_ADAPTER)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 34.5kB / 50.5MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Stage 1 non-instruction LoRA adapter pushed to:
saadtariq/pharma-tinyllama-non-instruction-lora-adapter


## Step 21: Stage 1.17: Reload base model + LoRA adapter correctly

In [62]:
# Clean old objects to free memory.

del trainer

try:
    del model
    del base_model
except NameError:
    pass

gc.collect()

if use_cuda:
    torch.cuda.empty_cache()

In [63]:
inference_tokenizer = AutoTokenizer.from_pretrained(
    non_instruction_config.adapter_dir, use_fast=True
)

if inference_tokenizer.pad_token is None:
    inference_tokenizer.pad_token = inference_tokenizer.eos_token

In [64]:
if use_cuda:
    inference_base_model = AutoModelForCausalLM.from_pretrained(
        non_instruction_config.model_name,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
else:
    inference_base_model = AutoModelForCausalLM.from_pretrained(
        non_instruction_config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [65]:
inference_model = PeftModel.from_pretrained(
    inference_base_model,
    non_instruction_config.adapter_dir
)

In [67]:
# inference_model.eval()

In [68]:
print("Base model + LoRA adapter loaded successfully for inference.")

Base model + LoRA adapter loaded successfully for inference.


## Step 22 - Stage 1.18: Inference helper

In [69]:
def generate_completion(prompt: str, max_new_tokens: int = 120) -> str:
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Convert prompt text into token IDs.
    inputs = inference_tokenizer(prompt, return_tensors="pt").to(device)

    # Generate text without calculating gradients because we are doing inference, not training.
    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=inference_tokenizer.eos_token_id,
        )

    # Convert generated token IDs back into readable text.
    return inference_tokenizer.decode(outputs[0], skip_special_tokens=True)

## Step 23 - Stage 1.19: Test text continuation

In [70]:
prompts = [
    "Metformin is one of the most widely prescribed oral antihyperglycemic agents",
    "Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe",
    "Artificial intelligence is transforming pharmaceutical research by accelerating",
]

In [71]:
for prompt in prompts:
    print("=" * 100)
    print("PROMPT:")
    print(prompt)
    print("\nMODEL CONTINUATION:")
    print(generate_completion(prompt, max_new_tokens=120))
    print()

PROMPT:
Metformin is one of the most widely prescribed oral antihyperglycemic agents

MODEL CONTINUATION:


[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Metformin is one of the most widely prescribed oral antihyperglycemic agents in the world. It is a sulfonylurea class drug, which has been used to treat type 2 diabetes for over 30 years. However, despite its proven efficacy and safety, the mechanism by which it works remains unknown. A recent study from the National Institutes of Health (NIH) reveals that Metformin can directly affect the cellular processes that regulate the production of insulin. This finding could lead to the development of new drugs targeting these processes.
"These findings are very important because they suggest that Met

PROMPT:
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe

MODEL CONTINUATION:


[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe, a statin drug that reduces cholesterol and LDL cholesterol, lowers LDL-C by 39% compared to statins alone.
The study was published online in the New England Journal of Medicine.
Atorvastatin is a lipid (fat) lowering drug used for primary prevention of cardiovascular disease, as well as secondary prevention of coronary artery disease, strokes and other types of heart attacks.
Atrial fibrillation is an irregular heartbeat that causes unpredict

PROMPT:
Artificial intelligence is transforming pharmaceutical research by accelerating

MODEL CONTINUATION:
Artificial intelligence is transforming pharmaceutical research by accelerating drug discovery, reducing time-to-market and increasing the efficiency of R&D operations.
Industrial IoT is driving the next wave of innovation in healthcare as a growing number of devices and apps are connected to create smart environments for better patient care.
MedTech IoT is revol

## Step 24 - Stage 1.20 Optional merge step

- This step merges the LoRA adapter into the base model.
- Use this only when you want a standalone model for deployment

In [73]:
# Reload the base model in float16 for safe merging.
base_model = AutoModelForCausalLM.from_pretrained(
    non_instruction_config.model_name,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [74]:
# Load the trained LoRA adapter on top of the base model.
model_with_adapter = PeftModel.from_pretrained(
    base_model,
    non_instruction_config.adapter_dir
)

In [75]:
# Merge LoRA adapter weights into the base model weights.
merged_model = model_with_adapter.merge_and_unload()

In [76]:
non_instruction_config.merged_model_dir

'/content/pharma_tinyllama_merged_model'

In [77]:
# Save the merged standalone model and tokenizer.

merged_model.save_pretrained(non_instruction_config.merged_model_dir)

inference_tokenizer.save_pretrained(non_instruction_config.merged_model_dir)

print(f"Merged model saved to: {non_instruction_config.merged_model_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to: /content/pharma_tinyllama_merged_model


# Stage 2: Continue with Instruction Fine-Tuning on the Same Domain-Adapted Finetuned Model

In Stage 1, we performed **non-instruction fine-tuning / domain-adaptive continued pretraining** on raw pharma PDF text.

Now we continue from the **same Stage 1 LoRA adapter** and perform **instruction fine-tuning** using structured pharma instruction-response examples.

```text
Base TinyLlama
   ↓
Stage 1: Raw pharma text continued pretraining using LoRA
   ↓
Stage 1 domain-adapted LoRA adapter
   ↓
Stage 2: Instruction fine-tuning on pharma Q&A data
   ↓
Final instruction-tuned pharma LoRA adapter
```

This means we are not starting from scratch. We are continuing from the model adapter trained in the previous stage.

What changes in instruction fine-tuning?

For non-instruction fine-tuning, the data looked like raw text:

```text
Metformin is one of the most widely prescribed oral antihyperglycemic agents...
```

For instruction fine-tuning, the data looks like:

```json
{
  "instruction": "Explain the mechanism of action of Metformin.",
  "input": "",
  "output": "Metformin primarily activates AMPK..."
}
```

This teaches the model not only pharma language, but also how to answer user instructions.

In [78]:
instruction_config.output_dir

'/content/pharma_tinyllama_instruction_lora_output'

In [79]:
print(json.dumps(asdict(instruction_config), indent=2))

{
  "instruction_data_path": "/content/pharma_instruction_dataset.jsonl",
  "model_name": "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
  "output_dir": "/content/pharma_tinyllama_instruction_lora_output",
  "adapter_dir": "/content/pharma_tinyllama_instruction_lora_adapter",
  "merged_model_dir": "/content/pharma_tinyllama_instruction_merged_model",
  "max_length": 512,
  "test_size": 0.15,
  "seed": 42,
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "num_train_epochs": 5.0,
  "per_device_train_batch_size": 1,
  "per_device_eval_batch_size": 1,
  "gradient_accumulation_steps": 8,
  "learning_rate": 0.0001,
  "warmup_steps": 5,
  "weight_decay": 0.01,
  "logging_steps": 1,
  "logging_first_step": true,
  "eval_steps": 1,
  "save_steps": 10,
  "save_total_limit": 2,
  "max_steps": -1
}


In [80]:
os.makedirs(instruction_config.output_dir, exist_ok=True)
os.makedirs(instruction_config.adapter_dir, exist_ok=True)
os.makedirs(instruction_config.merged_model_dir, exist_ok=True)

In [81]:
if not os.path.exists(instruction_config.instruction_data_path):
    print(f"Instruction dataset not found at: {instruction_config.instruction_data_path}")
else:
    print(f"Instruction dataset found: {instruction_config.instruction_data_path}")

Instruction dataset found: /content/pharma_instruction_dataset.jsonl


## Step 25 - Stage 2.1: Load Instruction Dataset

In [82]:
instruction_dataset = load_dataset(
    "json",
    data_files=instruction_config.instruction_data_path,
    split="train"
)

print(instruction_dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output', 'source_page', 'topic'],
    num_rows: 48
})


In [83]:
print(instruction_dataset[0])

{'instruction': 'Explain the primary mechanism of action of metformin.', 'input': '', 'output': 'Metformin primarily acts by activating AMP-activated protein kinase, also called AMPK. AMPK is a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while reducing hepatic gluconeogenesis, which helps lower blood glucose levels.', 'source_page': 1, 'topic': 'Metformin pharmacology'}


## Step 26 - Stage 2.2: Format Instruction Records

We convert every record into Alpaca-style training text:

```text
### Instruction:
<instruction text>

### Input:
<input text, only if present>

### Response:
<output text>
```


In [84]:
def format_instruction_record(record):
    instruction = str(record.get("instruction", "")).strip()
    input_text = str(record.get("input", "")).strip()
    output_text = str(record.get("output", "")).strip()

    if input_text:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n{output_text}"
        )
    else:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Response:\n{output_text}"
        )

    return {"text": text}

In [85]:
instruction_dataset = instruction_dataset.map(format_instruction_record)

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

In [86]:
instruction_dataset

Dataset({
    features: ['instruction', 'input', 'output', 'source_page', 'topic', 'text'],
    num_rows: 48
})

In [87]:
print(instruction_dataset[0]["text"])

### Instruction:
Explain the primary mechanism of action of metformin.

### Response:
Metformin primarily acts by activating AMP-activated protein kinase, also called AMPK. AMPK is a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while reducing hepatic gluconeogenesis, which helps lower blood glucose levels.


## Step 27 - Stage 2.3: Create train-validation split

In [88]:
instruction_datasets = instruction_dataset.train_test_split(
    test_size=0.15,
    seed=42
)

instruction_datasets["validation"] = instruction_datasets.pop("test")

print(instruction_datasets)
print("Train examples:", len(instruction_datasets["train"]))
print("Validation examples:", len(instruction_datasets["validation"]))

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'source_page', 'topic', 'text'],
        num_rows: 40
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output', 'source_page', 'topic', 'text'],
        num_rows: 8
    })
})
Train examples: 40
Validation examples: 8


## Step 28 - Stage 2.4: Load Tokenizer

In [89]:
tokenizer = AutoTokenizer.from_pretrained(
    instruction_config.model_name,
    use_fast=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(tokenizer.pad_token)

</s>


## Step 29 - Stage 2.5: Tokenize the instruction dataset

When we tokenize instruction data, examples are padded to the same length (`max_length`). We don't want the model to learn from padding tokens, so we set their label to `-100`, which tells PyTorch to ignore those positions when calculating the loss.

In [90]:
def tokenize_instruction_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=instruction_config.max_length,
    )

    # For causal LM, labels are copied from input_ids.
    tokens["labels"] = tokens["input_ids"].copy()

    # Ignore padding tokens in the loss calculation.
    tokens["labels"] = [
        [
            token if mask == 1 else -100
            for token, mask in zip(input_ids, attention_mask)
        ]
        for input_ids, attention_mask in zip(tokens["input_ids"], tokens["attention_mask"])
    ]

    return tokens

In [91]:
instruction_tokenized_datasets = instruction_datasets.map(
    tokenize_instruction_function,
    batched=True,
    remove_columns=instruction_datasets["train"].column_names,
    desc="Tokenizing instruction dataset",
)

print(instruction_tokenized_datasets)

Tokenizing instruction dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Tokenizing instruction dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 40
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 8
    })
})


## Step 30 - Stage 2.6: Load merged Stage 1 model and add new LoRA adapter for instruction tuning

In [92]:
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

use_cuda = torch.cuda.is_available()
print(f"CUDA available: {use_cuda}")

CUDA available: True


In [93]:
non_instruction_config.merged_model_dir

'/content/pharma_tinyllama_merged_model'

In [94]:
if use_cuda:
    # Load merged Stage 1 model in 4-bit mode for QLoRA instruction tuning.
    instruction_base_model = AutoModelForCausalLM.from_pretrained(
        non_instruction_config.merged_model_dir,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )

    instruction_base_model = prepare_model_for_kbit_training(instruction_base_model)

else:
    # CPU fallback. Training on CPU will be slow.
    instruction_base_model = AutoModelForCausalLM.from_pretrained(
        non_instruction_config.merged_model_dir,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [95]:
instruction_base_model.config.use_cache = False

# Create a new LoRA adapter for instruction fine-tuning.
instruction_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

instruction_model = get_peft_model(
    instruction_base_model,
    instruction_lora_config
)

instruction_model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


## Step 31 - Stage 2.7: Instruction fine-tuning data collator

The `Trainer` needs a data collator to assemble tokenized examples into mini-batches. Since our labels are already prepared with `-100` masking, we use `DataCollatorForLanguageModeling` with `mlm=False` (causal LM, not masked LM).


In [96]:
instruction_data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

## Step 32 - Stage 2.8: Instruction fine-tuning arguments

In [97]:
instruction_config.output_dir

'/content/pharma_tinyllama_instruction_lora_output'

In [98]:
instruction_config.adapter_dir

'/content/pharma_tinyllama_instruction_lora_adapter'

In [99]:
instruction_training_args = TrainingArguments(
    output_dir=instruction_config.output_dir,

    num_train_epochs=instruction_config.num_train_epochs,
    max_steps=instruction_config.max_steps,

    per_device_train_batch_size=instruction_config.per_device_train_batch_size,
    per_device_eval_batch_size=instruction_config.per_device_eval_batch_size,
    gradient_accumulation_steps=instruction_config.gradient_accumulation_steps,

    learning_rate=instruction_config.learning_rate,
    warmup_steps=instruction_config.warmup_steps,
    weight_decay=instruction_config.weight_decay,

    logging_steps=instruction_config.logging_steps,
    logging_first_step=instruction_config.logging_first_step,

    eval_strategy="steps",
    eval_steps=instruction_config.eval_steps,

    save_steps=instruction_config.save_steps,
    save_total_limit=instruction_config.save_total_limit,

    fp16=use_cuda,
    bf16=False,

    report_to="none",
    remove_unused_columns=False,
)

In [101]:
# print(instruction_training_args)

## Step 33 - Stage 2.9: Build Instruction Trainer

In [102]:
instruction_trainer = Trainer(
    model=instruction_model,
    args=instruction_training_args,
    train_dataset=instruction_tokenized_datasets["train"],
    eval_dataset=instruction_tokenized_datasets["validation"],
    data_collator=instruction_data_collator,
)

print("Instruction Trainer is ready.")

Instruction Trainer is ready.


## Step 34 - Stage 2.10: Start instruction fine-tuning

In [103]:
instruction_train_result = instruction_trainer.train()

print("Instruction fine-tuning completed.")
print(instruction_train_result)

Step,Training Loss,Validation Loss
1,2.060840,2.299327
2,2.171909,2.281923
3,2.511540,2.244166
4,2.131257,2.189779
5,2.185680,2.120422
6,2.059405,2.041484
7,1.779725,1.969717
8,1.783992,1.901157
9,1.861670,1.834988
10,1.903324,1.777402


Instruction fine-tuning completed.
TrainOutput(global_step=25, training_loss=1.6864664602279662, metrics={'train_runtime': 177.8483, 'train_samples_per_second': 1.125, 'train_steps_per_second': 0.141, 'total_flos': 643355482521600.0, 'train_loss': 1.6864664602279662, 'epoch': 5.0})


## Step 35 - Stage 2.11: Save final instruction-tuned LoRA adapter

In [104]:
instruction_config.adapter_dir

'/content/pharma_tinyllama_instruction_lora_adapter'

In [105]:
instruction_trainer.model.save_pretrained(instruction_config.adapter_dir)
tokenizer.save_pretrained(instruction_config.adapter_dir)

print(f"Final instruction-tuned LoRA adapter saved to: {instruction_config.adapter_dir}")
print(os.listdir(instruction_config.adapter_dir))

Final instruction-tuned LoRA adapter saved to: /content/pharma_tinyllama_instruction_lora_adapter
['adapter_config.json', 'adapter_model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'README.md']


## Step 36 - Stage 2.12: Push Stage 2 instruction LoRA adapter to Hugging Face

In [106]:
HF_REPO_INSTRUCTION_ADAPTER

'saadtariq/pharma-tinyllama-instruction-lora-adapter'

In [107]:
instruction_trainer.model.push_to_hub(
    HF_REPO_INSTRUCTION_ADAPTER,
    private=True
)

tokenizer.push_to_hub(
    HF_REPO_INSTRUCTION_ADAPTER,
    private=True
)

print("Stage 2 instruction LoRA adapter pushed to:")
print(HF_REPO_INSTRUCTION_ADAPTER)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 34.5kB / 50.5MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Stage 2 instruction LoRA adapter pushed to:
saadtariq/pharma-tinyllama-instruction-lora-adapter


## Step 37 - Stage 2.13: Reload final instruction-tuned adapter for inference

In [108]:
non_instruction_config.merged_model_dir

'/content/pharma_tinyllama_merged_model'

In [109]:
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

if use_cuda:
    base_model = AutoModelForCausalLM.from_pretrained(
        non_instruction_config.merged_model_dir,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        non_instruction_config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

final_instruction_model = PeftModel.from_pretrained(
    base_model,
    non_instruction_config.adapter_dir,
)

final_instruction_model.eval()

print("Final instruction-tuned model loaded successfully.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Final instruction-tuned model loaded successfully.


## Step 38 - Stage 2.14: Instruction-style inference helper

In [110]:
def build_instruction_prompt(instruction, input_text=""):
    instruction = instruction.strip()
    input_text = input_text.strip()

    if input_text:
        return (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n"
        )

    return (
        f"### Instruction:\n{instruction}\n\n"
        f"### Response:\n"
    )

In [111]:
def generate_instruction_response(instruction, input_text="", max_new_tokens=150):
    prompt = build_instruction_prompt(instruction, input_text)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(final_instruction_model.device)

    with torch.no_grad():
        outputs = final_instruction_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

## Step 39: Stage 2.15: Test instruction-tuned pharma model

In [112]:
test_questions = [
    "Explain the primary mechanism of action of metformin.",
    "Why can atorvastatin and ezetimibe reduce LDL-C more effectively together?",
    "Summarize the role of lipid nanoparticles in mRNA vaccines.",
    "Why should AI predictions in drug discovery be experimentally validated?",
]

for question in test_questions:
    print("=" * 100)
    print("QUESTION:")
    print(question)

    print("\nMODEL RESPONSE:")
    print(generate_instruction_response(question, max_new_tokens=150))

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
Explain the primary mechanism of action of metformin.

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Explain the primary mechanism of action of metformin.

### Response:
Metformin is a biguanide, which has antidiabetic effects. It is a potent inhibitor of 2-Amino-3-(4-hydroxyphenyl) pyridine reductase (HAPR), which converts insulin to glucose. Metformin acts by preventing the conversion of insulin into glucose and also by inhibiting glycogen synthesis. Metformin is a bile acid sequestrant, which prevents the absorption of cholesterol from the intestinal tract. In addition, it suppresses the growth of tumors, such as breast cancer cells. Metformin induces apoptosis,
QUESTION:
Why can atorvastatin and ezetimibe reduce LDL-C more effectively together?

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Why can atorvastatin and ezetimibe reduce LDL-C more effectively together?

### Response:
The reason why these two drugs work well together is because they target different pathways in the body. 
Atorvastatin blocks a pathway that leads to the production of cholesterol, while ezetimibe helps remove cholesterol from blood by blocking a different pathway.

### What does this mean for your patients?
This means that patients with high triglycerides and low HDL-C who take statins (either alone or in combination) will be able to lower their triglycerides even further.
It also means that patients with high LDL-C levels who take ezetimibe will have their LDL-C even higher.

QUESTION:
Summarize the role of lipid nanoparticles in mRNA vaccines.

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Summarize the role of lipid nanoparticles in mRNA vaccines.

### Response:
The role of lipid nanoparticles is to increase the stability and accessibility of mRNA vaccine to the body’s cells, allowing it to be taken up into cells for transcription and translation. This increases the chance of an antibody response to develop from the immune system. The vaccine itself does not need to be stored at cold temperatures because the lipid nanoparticles provide a more stable environment for the vaccine to be kept in. In addition, the lipid nanoparticles have been shown to help mRNA vaccines last longer in storage by keeping them from oxidizing or breaking down.

### Additional Resources:
[Nanotechnology in
QUESTION:
Why should AI predictions in drug discovery be experimentally validated?

MODEL RESPONSE:
### Instruction:
Why should AI predictions in drug discovery be experimentally validated?

### Response:
This is a very important question to ask because if AI predictions are n

## Step 40: Stage 2.16: Merge instruction-tuned LoRA adapter into base model

In [113]:
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [114]:
instruction_config.merged_model_dir

'/content/pharma_tinyllama_instruction_merged_model'

In [115]:
# Load the original base model in normal precision for safe merging.
base_model_for_merge = AutoModelForCausalLM.from_pretrained(
    non_instruction_config.merged_model_dir,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [116]:
# Load the tokenizer.
tokenizer_for_merge = AutoTokenizer.from_pretrained(
    non_instruction_config.model_name,
    trust_remote_code=True,
)

if tokenizer_for_merge.pad_token is None:
    tokenizer_for_merge.pad_token = tokenizer_for_merge.eos_token

In [117]:
# Attach the final instruction-tuned LoRA adapter.
model_with_instruction_adapter = PeftModel.from_pretrained(
    base_model_for_merge,
    instruction_config.adapter_dir,
)

In [118]:
# Merge LoRA adapter weights into the base model weights.
merged_instruction_model = model_with_instruction_adapter.merge_and_unload()

In [119]:
instruction_config.merged_model_dir

'/content/pharma_tinyllama_instruction_merged_model'

In [120]:
# Save the standalone merged model and tokenizer.
merged_instruction_model.save_pretrained(instruction_config.merged_model_dir)
tokenizer_for_merge.save_pretrained(instruction_config.merged_model_dir)

print(f"Merged instruction-tuned model saved to: {instruction_config.merged_model_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged instruction-tuned model saved to: /content/pharma_tinyllama_instruction_merged_model


# Stage 3: Preference Tuning with DPO

In Stage 1, we adapted the model to the pharma domain using raw non-instruction text.

In Stage 2, we instruction-tuned the model using instruction-response data.

In Stage 3, we will use **preference data** with DPO.

DPO data has three main columns:

```text
prompt
chosen
rejected
```

- `prompt` is the user instruction.
- `chosen` is the preferred/better answer.
- `rejected` is the weaker answer.

The goal of DPO is to make the model prefer the `chosen` response over the `rejected` response.

Paper link: https://arxiv.org/pdf/2305.18290

| Fine-tuning stage         | Data format                 | What model learns?              |
| ------------------------- | --------------------------- | ----------------------------------- |
| **Non-instruction FT**    | Raw text                    | Domain language aur knowledge style |
| **Instruction FT**        | Instruction → Response      | Answers based on user instructions     |
| **DPO Preference Tuning** | Prompt → Chosen vs Rejected | Prefer better answer          |

| Section                       | Simple Meaning                                                                                         | Key Point                                                                           |
| ----------------------------- | ------------------------------------------------------------------------------------------------------ | ----------------------------------------------------------------------------------- |
| **DPO Full Form**             | Direct Preference Optimization                                                                         | The model is trained directly using preference data.                                |
| **Paper**                     | “Direct Preference Optimization”                                                                       | Published at NeurIPS 2023 by Stanford researchers.                                  |
| **Main Idea**                 | Teach the model which answer is better and which answer is weaker.                                     | The model learns to prefer the `chosen` answer and avoid the `rejected` answer.     |
| **Before DPO: RLHF**          | RLHF usually has three stages.                                                                         | SFT → Reward Model → PPO                                                            |
| **RLHF Problem**              | RLHF is complex, expensive, and unstable.                                                              | Training a reward model and using PPO require high compute and careful tuning.      |
| **DPO Insight**               | A separate reward model is not required.                                                               | The language model itself can act like an implicit reward model.                    |
| **DPO Dataset Format**        | Each sample has three main fields.                                                                     | `prompt`, `chosen`, and `rejected`                                                  |
| **Prompt**                    | The user question or instruction.                                                                      | Example: “Explain the mechanism of metformin.”                                      |
| **Chosen**                    | The better or preferred answer.                                                                        | Usually accurate, complete, safe, and well-structured.                              |
| **Rejected**                  | The weaker or rejected answer.                                                                         | Usually vague, incomplete, incorrect, or unsafe.                                    |
| **DPO Training Goal**         | Increase the probability of the preferred answer.                                                      | The model becomes more likely to generate answers like the `chosen` response.       |
| **Role of Rejected Answer**   | Shows the model what type of answer to avoid.                                                          | The model reduces the probability of the `rejected` style answer.                   |
| **Reference Model**           | Usually the SFT model.                                                                                 | It prevents the DPO model from drifting too far from the original fine-tuned model. |
| **Policy Model**              | The model being trained during DPO.                                                                    | It learns to prefer the `chosen` answer over the `rejected` answer.                 |
| **Beta β**                    | A control parameter.                                                                                   | It controls how strongly the model moves away from the reference model.             |
| **DPO Loss**                  | A binary classification-style loss.                                                                    | It trains the model to make the `chosen` answer win over the `rejected` answer.     |
| **Reward Model Needed?**      | No.                                                                                                    | DPO removes the need for separate reward model training.                            |
| **PPO Needed?**               | No.                                                                                                    | DPO works more like supervised training instead of reinforcement learning.          |
| **Sampling During Training?** | No.                                                                                                    | DPO does not require an expensive generation loop like PPO.                         |
| **Main Advantage**            | Simpler and more stable.                                                                               | Easier to implement compared to traditional RLHF.                                   |
| **Training Cost**             | Lower than RLHF.                                                                                       | Only the policy model is trained.                                                   |
| **DPO vs SFT**                | SFT teaches the model how to answer.                                                                   | DPO teaches the model which answer is better.                                       |
| **DPO vs RLHF**               | RLHF uses a reward model and PPO.                                                                      | DPO directly uses preference loss.                                                  |
| **Gradient Intuition**        | Stronger updates happen when the model ranks answers incorrectly.                                      | The model learns more from difficult examples.                                      |
| **Practical Pipeline**        | Start with an SFT model, add preference data, then train with DPO.                                     | This creates a simple alignment pipeline.                                           |
| **Common Beta Value**         | The paper commonly used `β = 0.1`.                                                                     | For summarization tasks, `β = 0.5` was also used.                                   |
| **Experiments**               | Tested on sentiment, summarization, and dialogue tasks.                                                | DPO can perform equal to or better than PPO.                                        |
| **Limitation**                | Large-scale training, reward hacking, and out-of-distribution generalization are still open questions. | DPO is powerful, but not perfect.                                                   |
| **Classroom One-Liner**       | DPO teaches the model which answer is better.                                                          | Instruction tuning teaches answering; DPO teaches preference.                       |


## Step 41 - Stage 3.1: Load DPO Preference Dataset

In [121]:
preference_config.preference_data_path

'/content/pharma_preference_dataset.jsonl'

In [122]:
preference_dataset = load_dataset(
    "json",
    data_files=preference_config.preference_data_path,
    split="train"
)

Generating train split: 0 examples [00:00, ? examples/s]

In [123]:
print(preference_dataset)

Dataset({
    features: ['prompt', 'chosen', 'rejected', 'source_page', 'topic'],
    num_rows: 48
})


In [124]:
print(preference_dataset[0])

{'prompt': '### Instruction:\nExplain the primary mechanism of action of metformin.\n\n### Response:\n', 'chosen': 'Metformin primarily acts by activating AMP-activated protein kinase, also called AMPK. AMPK is a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while reducing hepatic gluconeogenesis, which helps lower blood glucose levels.', 'rejected': 'Metformin mainly works by increasing insulin secretion from the pancreas, and kidney function is usually not very relevant. Its side effects are generally not important unless the patient feels very sick.', 'source_page': 1, 'topic': 'Metformin pharmacology'}


## Stage 42 - Step 3.2: Create train-validation split

In [125]:
preference_dataset = preference_dataset.train_test_split(
    test_size=0.15,
    seed=42
)

In [126]:
# Rename test split to validation split
preference_dataset["validation"] = preference_dataset.pop("test")

In [127]:
print("After train-validation split:")
print(preference_dataset)
print("Train rows:", len(preference_dataset["train"]))
print("Validation rows:", len(preference_dataset["validation"]))

After train-validation split:
DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'source_page', 'topic'],
        num_rows: 40
    })
    validation: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'source_page', 'topic'],
        num_rows: 8
    })
})
Train rows: 40
Validation rows: 8


## Step 43 - Stage 3.3: Load merged instruction model as base for preference tuning and Attach a LoRA Adapter

In [128]:
instruction_config.merged_model_dir

'/content/pharma_tinyllama_instruction_merged_model'

In [129]:
use_cuda = torch.cuda.is_available()

In [130]:
if use_cuda:
    preference_base_model = AutoModelForCausalLM.from_pretrained(
        instruction_config.merged_model_dir,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )

    preference_base_model = prepare_model_for_kbit_training(preference_base_model)

else:
    preference_base_model = AutoModelForCausalLM.from_pretrained(
        instruction_config.merged_model_dir,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [131]:
preference_base_model.config.use_cache = False

In [132]:
# Create a new LoRA adapter for preference tuning.
preference_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

In [133]:
preference_model = get_peft_model(
    preference_base_model,
    preference_lora_config,
)

preference_model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


## Step 44: Stage 3.4: Configure DPO training

`DPOConfig` extends `TrainingArguments` with DPO-specific fields:

- `beta`: how strongly the model is pushed toward `chosen` over `rejected`, relative to the reference model.
- `max_length` / `max_prompt_length`: truncation limits for the full sequence and for the prompt portion.

Since `ref_model=None` is passed to `DPOTrainer` below, TRL will automatically use a frozen copy of the policy model's base weights as the reference model (this works cleanly with a LoRA adapter, since the adapter can be disabled internally to recover the reference behavior).

In [134]:
dpo_training_args = DPOConfig(
    output_dir=preference_config.output_dir,

    # Training duration.
    num_train_epochs=preference_config.num_train_epochs,
    max_steps=preference_config.max_steps,

    # Batch settings.
    per_device_train_batch_size=preference_config.per_device_train_batch_size,
    per_device_eval_batch_size=preference_config.per_device_eval_batch_size,
    gradient_accumulation_steps=preference_config.gradient_accumulation_steps,

    # Optimizer settings.
    learning_rate=preference_config.learning_rate,
    warmup_steps=preference_config.warmup_steps,
    weight_decay=preference_config.weight_decay,

    # Logging and evaluation.
    logging_steps=preference_config.logging_steps,
    logging_first_step=preference_config.logging_first_step,
    eval_strategy="steps",
    eval_steps=preference_config.eval_steps,

    # Checkpoint saving.
    save_steps=preference_config.save_steps,
    save_total_limit=preference_config.save_total_limit,

    # Precision settings.
    fp16=False,
    bf16=False,

    # Disable external logging tools.
    report_to="none",

    # Keep required columns.
    remove_unused_columns=False,

    # DPO hyperparameters.
    beta=preference_config.beta,
)

## Stage 45 - Step 3.5: Build DPO Trainer

In [135]:
dpo_trainer = DPOTrainer(
    model=preference_model,
    ref_model=None,  # None means TRL will internally use the reference behavior
    args=dpo_training_args,

    train_dataset=preference_dataset["train"],
    eval_dataset=preference_dataset["validation"],

    processing_class=tokenizer,
)

print("DPOTrainer is ready.")

Adding EOS to train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Dropping fully truncated examples from eval dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

DPOTrainer is ready.


## Stage 46 - Stage 3.6: Start DPO preference tuning

| Parameter               | Short Meaning                                                                              |
| ----------------------- | ------------------------------------------------------------------------------------------ |
| **Step**                | Current optimizer step during training.                                                    |
| **Training Loss**       | DPO loss on the training data; lower is generally better.                                  |
| **Validation Loss**     | DPO loss on unseen validation data; helps check generalization.                            |
| **Entropy**             | Measures how uncertain the model is; higher means more random, lower means more confident. |
| **Num Tokens**          | Total number of tokens processed so far.                                                   |
| **Logits/chosen**       | Raw model score for the preferred answer.                                                  |
| **Logits/rejected**     | Raw model score for the rejected answer.                                                   |
| **Mean Token Accuracy** | Average token-level prediction accuracy.                                                   |
| **Rewards/chosen**      | DPO implicit reward for the preferred answer; should be higher.                            |
| **Rewards/rejected**    | DPO implicit reward for the rejected answer; should be lower.                              |
| **Rewards/accuracies**  | How often the model ranks the chosen answer above the rejected answer.                     |
| **Rewards/margins**     | Difference between chosen reward and rejected reward; positive is good.                    |
| **Logps/chosen**        | Log probability of the chosen answer; less negative means more likely.                     |
| **Logps/rejected**      | Log probability of the rejected answer; ideally more negative than chosen.                 |


Simple summary: In DPO training, the main goal is to make the model assign higher probability and higher reward to the chosen answer than the rejected answer.

In [136]:
dpo_train_result = dpo_trainer.train()

print("DPO preference tuning completed.")
print(dpo_train_result)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
1,0.693147,0.693147,1.872145,679.000000,-3.383511,-3.498676,0.601007,0.000000,0.000000,0.000000,0.000000,-117.141771,-123.461451
2,0.693147,0.653907,1.871173,1413.000000,-3.380322,-3.497383,0.601007,0.015193,-0.066026,1.000000,0.081219,-116.989846,-124.121712
3,0.619596,0.558889,1.865429,2099.000000,-3.371231,-3.492360,0.602144,0.068225,-0.231495,1.000000,0.299720,-116.459523,-125.776404
4,0.517615,0.481193,1.859287,2789.000000,-3.360098,-3.485303,0.600968,0.114503,-0.396678,1.000000,0.511181,-115.996746,-127.428230
5,0.497758,0.397676,1.847964,3431.000000,-3.345835,-3.475785,0.603017,0.169626,-0.610851,1.000000,0.780477,-115.445508,-129.569962
6,0.399175,0.317702,1.837259,4051.000000,-3.330060,-3.464749,0.603288,0.230460,-0.872199,1.000000,1.102659,-114.837172,-132.183436
7,0.276943,0.253438,1.822029,4703.000000,-3.310148,-3.450386,0.599373,0.283361,-1.158365,1.000000,1.441726,-114.308164,-135.045103
8,0.225683,0.200698,1.805807,5380.000000,-3.288241,-3.433649,0.593675,0.329536,-1.456906,1.000000,1.786442,-113.846411,-138.030515
9,0.107595,0.157900,1.789193,6052.000000,-3.264600,-3.415805,0.597952,0.360090,-1.775109,1.000000,2.135198,-113.540875,-141.212536
10,0.152488,0.122420,1.774048,6698.000000,-3.241656,-3.398883,0.595257,0.356370,-2.132690,1.000000,2.489060,-113.578071,-144.788353


DPO preference tuning completed.
TrainOutput(global_step=30, training_loss=0.1459912049516182, metrics={'train_runtime': 160.5215, 'train_samples_per_second': 0.748, 'train_steps_per_second': 0.187, 'total_flos': 147834252288000.0, 'train_loss': 0.1459912049516182, 'epoch': 3.0})


## Step 47 - Stage 3.7: Save DPO preference-tuned LoRA adapter

In [137]:
preference_config.adapter_dir

'/content/pharma_tinyllama_preference_dpo_lora_adapter'

In [138]:
dpo_trainer.model.save_pretrained(preference_config.adapter_dir)
tokenizer.save_pretrained(preference_config.adapter_dir)

print(f"Preference-tuned LoRA adapter saved to: {preference_config.adapter_dir}")
print(os.listdir(preference_config.adapter_dir))

Preference-tuned LoRA adapter saved to: /content/pharma_tinyllama_preference_dpo_lora_adapter
['adapter_config.json', 'adapter_model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'README.md', 'ref']


## Step 48 - Stage 3.8: Push Stage 3 DPO LoRA adapter to Hugging Face

In [140]:
dpo_trainer.model.push_to_hub(
    HF_REPO_DPO_ADAPTER,
    private=True
)

tokenizer.push_to_hub(
    HF_REPO_DPO_ADAPTER,
    private=True
)

print("Stage 3 DPO LoRA adapter pushed to:")
print(HF_REPO_DPO_ADAPTER)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 34.5kB / 50.5MB            

  ...adapter_model.safetensors:   1%|1         |  342kB / 25.3MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Stage 3 DPO LoRA adapter pushed to:
saadtariq/pharma-tinyllama-dpo-lora-adapter


## Step 49 - Stage 3.9; Reload preference-tuned model for inference

In [141]:
instruction_config.merged_model_dir

'/content/pharma_tinyllama_instruction_merged_model'

In [142]:
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

if use_cuda:
    preference_inference_base_model = AutoModelForCausalLM.from_pretrained(
        instruction_config.merged_model_dir,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
else:
    preference_inference_base_model = AutoModelForCausalLM.from_pretrained(
        instruction_config.merged_model_dir,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

preference_inference_model = PeftModel.from_pretrained(
    preference_inference_base_model,
    preference_config.adapter_dir,
)

preference_inference_model.eval()

print("Preference-tuned model loaded successfully for inference.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Preference-tuned model loaded successfully for inference.


## Step 50 - Stage 3.10: Preference-tuned inference helper

In [143]:
def build_preference_prompt(instruction, input_text=""):
    instruction = instruction.strip()
    input_text = input_text.strip()

    if input_text:
        return (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n"
        )

    return (
        f"### Instruction:\n{instruction}\n\n"
        f"### Response:\n"
    )

In [144]:
def generate_preference_response(instruction, input_text="", max_new_tokens=150):
    prompt = build_preference_prompt(instruction, input_text)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(preference_inference_model.device)

    with torch.no_grad():
        outputs = preference_inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

## Step 51 - Stage 3.11: Test preference-tuned pharma model

In [145]:
preference_test_questions = [
    "Explain the primary mechanism of action of metformin.",
    "Why should AI predictions in drug discovery be experimentally validated?",
    "Define pharmacovigilance.",
    "Explain why pharmacovigilance continues after drug approval.",
]

for question in preference_test_questions:
    print("=" * 100)
    print("QUESTION:")
    print(question)

    print("\nMODEL RESPONSE:")
    print(generate_preference_response(question, max_new_tokens=150))

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
Explain the primary mechanism of action of metformin.

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Explain the primary mechanism of action of metformin.

### Response:
The primary mechanism of action of metformin is to decrease glucose production by inhibiting gluconeogenesis and glycogenolysis, reduce hepatic glucose production, decrease insulin resistance, and increase insulin sensitivity.
QUESTION:
Why should AI predictions in drug discovery be experimentally validated?

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Why should AI predictions in drug discovery be experimentally validated?

### Response:
Why should AI predictions in drug discovery be experimentally validated? AI predictions may overestimate the potential of a compound to be a therapeutic target, leading to premature approval or non-approval of a potential new compound. Experimental validation may include assessing whether AI prediction is sensitive to interfering factors such as genetic variation, protein-ligand binding affinity, and metabolism, and whether it is robust across multiple compound classes and different drug targets.
QUESTION:
Define pharmacovigilance.

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Define pharmacovigilance.

### Response:
Pharmacovigilance defines the process of monitoring and evaluating adverse event reports to identify safety trends and possible risks associated with medications. Pharmacovigilance includes surveillance, review, analysis, reporting, and communication activities.
QUESTION:
Explain why pharmacovigilance continues after drug approval.

MODEL RESPONSE:
### Instruction:
Explain why pharmacovigilance continues after drug approval.

### Response:
Pharmacovigilance continues after drug approval to monitor for unexpected safety signals, identify potential new indications, detect adverse events following discontinuation of therapy, and assess risk-benefit ratios in postapproval settings. Pharmacovigilance activities may include monitoring adverse event reporting, surveillance of new clinical evidence, reviewing regulatory submissions, and conducting epidemiologic studies.


## Step 52 - Stage 3.12: Optional: Merge DPO preference adapter into the instruction-tuned base model

In [146]:
preference_config.merged_model_dir

'/content/pharma_tinyllama_preference_merged_model'

In [147]:
instruction_config.merged_model_dir

'/content/pharma_tinyllama_instruction_merged_model'

In [148]:
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [149]:
# Load the merged instruction model in normal precision for safe merging.
base_model_for_preference_merge = AutoModelForCausalLM.from_pretrained(
    instruction_config.merged_model_dir,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [150]:
# Attach the DPO preference LoRA adapter.
model_with_preference_adapter = PeftModel.from_pretrained(
    base_model_for_preference_merge,
    preference_config.adapter_dir,
)

In [151]:
# Merge the preference adapter into the instruction-tuned base model.
final_merged_preference_model = model_with_preference_adapter.merge_and_unload()

In [152]:
# Save final standalone model and tokenizer.
final_merged_preference_model.save_pretrained(preference_config.merged_model_dir)
tokenizer.save_pretrained(preference_config.merged_model_dir)

print(f"Final merged preference-tuned model saved to: {preference_config.merged_model_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final merged preference-tuned model saved to: /content/pharma_tinyllama_preference_merged_model
